# Notebook 2: CNN 역할 분류기

## 목표
EfficientNet-B3를 fine-tuning해서 슬라이드 역할을 분류하는 모델을 학습.

| 클래스 | 역할 |
|---|---|
| 0 | 표지 |
| 1 | 섹션헤더 |
| 2 | 본문 |
| 3 | 도표/시각자료 |
| 4 | 마무리 |

## 예상 소요 시간 (A100)
- 학습: 30~50분

## 0. Drive 마운트 및 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/dadeum_ml'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

## 1. 패키지 설치

In [ ]:
!pip install -q timm torchmetrics
print('설치 완료')

## 2. 데이터 로더

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ROLE_NAMES = ['표지', '섹션헤더', '본문', '도표/시각자료', '마무리']
NUM_CLASSES = 5
IMG_SIZE = 224


class SlideRoleDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        # 이미지 파일이 실제로 존재하는 것만 사용
        self.df = df[df['image_path'].apply(lambda p: Path(p).exists())].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = int(row['weak_label'])
        return img, label


# 데이터 증강 (학습용)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# 데이터 로드 및 분할
df = pd.read_csv(f'{LABELS_DIR}/weak_labels.csv')
print(f'전체 슬라이드: {len(df)}장')
print(df['role_name'].value_counts())

# 덱 단위로 train/val 분할 (슬라이드 단위 분할 시 데이터 누수 발생)
deck_ids = df['deck_id'].unique()
np.random.seed(42)
np.random.shuffle(deck_ids)

split = int(len(deck_ids) * 0.85)
train_decks = set(deck_ids[:split])
val_decks = set(deck_ids[split:])

train_df = df[df['deck_id'].isin(train_decks)].reset_index(drop=True)
val_df = df[df['deck_id'].isin(val_decks)].reset_index(drop=True)

print(f'\n학습 슬라이드: {len(train_df)}장 ({len(train_decks)}개 덱)')
print(f'검증 슬라이드: {len(val_df)}장 ({len(val_decks)}개 덱)')

In [ ]:
BATCH_SIZE = 128  # A100 40GB 기준
NUM_WORKERS = 4

train_dataset = SlideRoleDataset(train_df, train_transform)
val_dataset = SlideRoleDataset(val_df, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'학습 배치 수: {len(train_loader)}')
print(f'검증 배치 수: {len(val_loader)}')

## 3. EfficientNet-B3 모델 설계

In [ ]:
import timm
import torch.nn as nn


class SlideRoleClassifier(nn.Module):
    def __init__(self, num_classes: int = 5, dropout: float = 0.3):
        super().__init__()
        # EfficientNet-B3 pretrained on ImageNet
        self.backbone = timm.create_model(
            'efficientnet_b3',
            pretrained=True,
            num_classes=0,       # 분류 헤드 제거
            global_pool='avg',
        )
        feat_dim = self.backbone.num_features  # 1536

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)     # (B, 1536)
        logits = self.classifier(features)  # (B, 5)
        return logits

    def extract_features(self, x):
        """임베딩만 반환 — 추후 Isolation Forest 입력으로 사용"""
        return self.backbone(x)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SlideRoleClassifier(num_classes=NUM_CLASSES).to(device)

# 파라미터 수 확인
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'전체 파라미터: {total:,}')
print(f'학습 가능 파라미터: {trainable:,}')

## 4. 학습

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchmetrics import Accuracy
import time

# 클래스 불균형 처리 — 본문이 압도적으로 많음
class_counts = train_df['weak_label'].value_counts().sort_index().values
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

EPOCHS = 20
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
accuracy = Accuracy(task='multiclass', num_classes=NUM_CLASSES).to(device)

best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}


for epoch in range(EPOCHS):
    # 학습
    model.train()
    train_loss = 0.0
    t0 = time.time()

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    train_loss /= len(train_loader)

    # 검증
    model.eval()
    val_loss = 0.0
    accuracy.reset()

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            val_loss += criterion(logits, labels).item()
            accuracy.update(logits, labels)

    val_loss /= len(val_loader)
    val_acc = accuracy.compute().item()
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
          f'train_loss={train_loss:.4f} | '
          f'val_loss={val_loss:.4f} | '
          f'val_acc={val_acc:.4f} | '
          f'{elapsed:.1f}s')

    # 최고 모델 저장
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc,
        }, f'{MODELS_DIR}/role_classifier_best.pt')
        print(f'  → 최고 모델 저장 (val_acc={val_acc:.4f})')

print(f'\n학습 완료. 최고 검증 정확도: {best_val_acc:.4f}')

In [ ]:
# 학습 곡선 시각화
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curve')
ax1.legend()

ax2.plot(history['val_acc'], label='Val Accuracy', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/training_curves.png', dpi=100)
plt.show()

## 5. CNN 임베딩으로 역할별 Feature 추출

학습된 EfficientNet-B3 백본으로 전체 슬라이드의 1536차원 임베딩을 추출.  
이 임베딩을 Notebook 4에서 역할별 Isolation Forest 입력으로 사용.

In [ ]:
# 최고 모델 로드
checkpoint = torch.load(f'{MODELS_DIR}/role_classifier_best.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'모델 로드 완료 (val_acc={checkpoint["val_acc"]:.4f})')


# 전체 데이터셋 임베딩 추출
full_dataset = SlideRoleDataset(df, val_transform)
full_loader = DataLoader(full_dataset, batch_size=256,
                         shuffle=False, num_workers=4, pin_memory=True)

all_embeddings = []
all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in tqdm(full_loader, desc='임베딩 추출'):
        imgs = imgs.to(device)
        embeddings = model.extract_features(imgs)  # (B, 1536)
        logits = model.classifier(embeddings)
        preds = logits.argmax(dim=1)

        all_embeddings.append(embeddings.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())

embeddings_np = np.concatenate(all_embeddings, axis=0)  # (N, 1536)
preds_np = np.concatenate(all_preds, axis=0)
labels_np = np.concatenate(all_labels, axis=0)

# 예측 역할을 df에 추가
valid_df = full_dataset.df.copy()
valid_df['pred_label'] = preds_np
valid_df['pred_role'] = [ROLE_NAMES[p] for p in preds_np]

# 임베딩 저장
np.save(f'{LABELS_DIR}/embeddings.npy', embeddings_np)
valid_df.to_csv(f'{LABELS_DIR}/labeled_with_preds.csv', index=False)

print(f'임베딩 저장 완료: {embeddings_np.shape}')

# 예측 정확도 확인
acc = (preds_np == labels_np).mean()
print(f'전체 정확도: {acc:.4f}')

# 혼동 행렬
from sklearn.metrics import confusion_matrix, classification_report
print('\n분류 리포트:')
print(classification_report(labels_np, preds_np, target_names=ROLE_NAMES))

In [ ]:
print('=== Notebook 2 완료 ===')
print(f'role_classifier_best.pt 저장 완료')
print(f'embeddings.npy 저장 완료: {embeddings_np.shape}')
print(f'최고 검증 정확도: {best_val_acc:.4f}')
print('\nNotebook 3 (HMM 학습)으로 이동하세요.')